# 19 — Paket review G3 saja untuk subset p025/p050 (CPU)
Keputusan pengguna 28 Sep 2026 (opsi A): hanya kandidat pemenang (G3) yang direview pada subset; arm synthetic_G0 subset ditiadakan. Key privat dibaca di kernel ini untuk memilih baris G3 dan tidak pernah disalin ke output. Reviewer mengetahui semua gambar berasal dari G3.


In [ ]:
from pathlib import Path
import csv, json, shutil, subprocess, sys, zipfile, hashlib
# CPU only. Input: notebook 18 output (subset study with private review keys) + code dataset (review tool).
# Output: G3-only review folders for p025 and p050, packed <=45 MB. Keys are read here and never copied out.
INPUT = Path('/kaggle/input'); WORK = Path('/kaggle/working'); OUT = WORK/'review_g3'; PACK = WORK/'review_g3_pack'
OUT.mkdir(); PACK.mkdir()
studies = sorted({p.parent for p in INPUT.rglob('study.json') if (p.parent/'p025/review_key_PRIVATE.json').exists()})
if len(studies) != 1: raise ValueError(f'expected one subset study; found {studies}')
STUDY = studies[0]
tools = sorted(INPUT.rglob('make_texture_review_tool.py'))
if not tools: raise ValueError('review tool not found; attach the code dataset')
TOOL = tools[0]
winner = json.loads((STUDY/'selection.json').read_text())['winner']
summary = {}
for fraction in (25, 50):
    scope = STUDY/f'p{fraction:03d}'
    key = json.loads((scope/'review_key_PRIVATE.json').read_text())
    keep = {r['blind_id'] for r in key if r['arm'] == winner}
    rows = list(csv.DictReader((scope/'review/review.csv').open(newline='')))
    fields = list(rows[0])
    target = OUT/f'p{fraction:03d}/review'; (target/'images').mkdir(parents=True)
    kept = [r for r in rows if r['blind_id'] in keep]
    if len(kept) != len(keep): raise ValueError('review.csv and key disagree')
    for r in kept: shutil.copyfile(scope/'review/images'/f"{r['blind_id']}.png", target/'images'/f"{r['blind_id']}.png")
    with (target/'review.csv').open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(kept)
    subprocess.run([sys.executable, str(TOOL), '--review', str(target)], check=True)
    summary[fraction] = dict(arm=winner, items=len(kept), all_items=len(rows),
                             review_csv_sha256=hashlib.sha256((target/'review.csv').read_bytes()).hexdigest())
files = sorted(p for p in OUT.rglob('*') if p.is_file())
if any('PRIVATE' in p.name for p in files): raise ValueError('private key in output')
parts, current, size = [], [], 0
for p in files:
    s = p.stat().st_size
    if current and size + s > 45*2**20: parts.append(current); current, size = [], 0
    current.append(p); size += s
parts.append(current)
manifest = dict(summary=summary, files={}, parts={})
for i, part in enumerate(parts):
    target = PACK/f'review_g3_part{i:02d}.zip'
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_STORED) as z:
        for p in part:
            rel = str(p.relative_to(OUT)); z.write(p, rel)
            manifest['files'][rel] = hashlib.sha256(p.read_bytes()).hexdigest()
    manifest['parts'][target.name] = dict(n=len(part), bytes=target.stat().st_size,
                                          sha256=hashlib.sha256(target.read_bytes()).hexdigest())
(PACK/'review_g3_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
shutil.rmtree(OUT)
print(json.dumps(summary, indent=1), {k: round(v['bytes']/2**20, 1) for k, v in manifest['parts'].items()})
